# Clinical-trial Text-to-SQL: context → serving → optimization → portability

## 0. Executive summary

One clinical application, one benchmark harness, interchangeable serving adapters.
The table updates as experiments run. **Saved cell outputs are retained. By default,
saved measurements are loaded and live model calls/deployments stay disabled.**
Quality here means execution-result match; reviewed semantic correctness is shown separately.
Earlier MVP results remain historical and are not silently imported as new measurements.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import Markdown, display
from clinical_trial_mvp.tutorial.harness import TutorialSession
session = TutorialSession(ROOT)
session.load_saved()  # Only measured tutorial records from this database, if present.
summary_view = display(Markdown(session.summary()), display_id=True)


| Experiment | Hardware | Runtime | Quality | TTFT | tok/s | Peak memory |
|---|---|---|---|---|---|---|
| MVP1/full · plan_only | Mac M4 Pro | llama_cpp | — | — | — | — |
| MVP1/retrieved · plan_only | Mac M4 Pro | llama_cpp | — | — | — | — |
| MVP2/llama_cpp · plan_only | Mac M4 Pro | llama_cpp | — | — | — | — |
| MVP2/ollama · plan_only | Mac M4 Pro | ollama | — | — | — | — |
| MVP3/vllm/c1 · completed | Dell Precision 7740 / Quadro RTX 5000 16 GB | vllm | 2/4 match | 0.19 | 40.67 | — |
| MVP3/vllm/c2 · completed | Dell Precision 7740 / Quadro RTX 5000 16 GB | vllm | 2/4 match | 0.24 | 67.09 | — |
| MVP3/vllm/c4 · completed | Dell Precision 7740 / Quadro RTX 5000 16 GB | vllm | 2/4 match | 0.37 | 105.37 | — |
| MVP3/sglang/c1 · plan_only | Dell RTX 5000 (verify exact SKU) | sglang | — | — | — | — |
| MVP3/sglang/c2 · plan_only | Dell RTX 5000 (verify exact SKU) | sglang | — | — | — | — |
| MVP3/sglang/c4 · plan_only | Dell RTX 5000 (verify exact SKU) | sglang | — | — | — | — |
| MVP4/NIM · skipped | Dell RTX 5000 (verify exact SKU) | nim | — | — | — | — |
| Optimization/full · plan_only | Dell Precision 7740 / Quadro RTX 5000 16 GB | vllm | — | — | — | — |
| Optimization/retrieved · plan_only | Dell Precision 7740 / Quadro RTX 5000 16 GB | vllm | — | — | — | — |
| Optimization/quantized · plan_only | Dell Precision 7740 / Quadro RTX 5000 16 GB | vllm | — | — | — | — |
| Optimization/prefix-cache · plan_only | Dell Precision 7740 / Quadro RTX 5000 16 GB | vllm | — | — | — | — |
| Optimization/concurrency-4 · plan_only | Dell Precision 7740 / Quadro RTX 5000 16 GB | vllm | — | — | — | — |
| ExecuTorch/edge · skipped | local CPU / XNNPACK | executorch | — | — | — | — |

## 1. Goal, workflow and setup

> How do schema selection, serving runtime and inference optimization affect the
> quality and performance of local clinical-trial Text-to-SQL?

**Clinical question → full/retrieved schema text → LLM plan/SQL → SQL validation → DuckDB.**

- **Mac M4 Pro:** same GGUF and quantization through llama.cpp server and Ollama.
- **Dell RTX 5000:** same frozen Hugging Face weights, tokenizer and supported precision
  through vLLM and SGLang. Confirm the exact GPU SKU/VRAM first; do not assume BF16 or NIM support.
- **Workload:** four fixed questions; seed 42; one repair; identical token cap.
  Serial clinical quality and concurrent serving are separate phases of the same harness.
- **Metrics:** SQL execution, result match, reviewed semantics, prompt tokens, first-content
  TTFT, p50/p95 latency, aggregate output tok/s, and sampled memory. Small-sample p95 is descriptive.

Run `python scripts/tutorial/setup.py --install` in a prepared Python 3.11 environment.
Installation, optional data rebuild, server launches and model export stay in
[scripts and the tutorial guide](../tutorial/README.md).


In [2]:
import json
import platform
import duckdb
from clinical_trial_mvp.tutorial.harness import (
    serve_and_benchmark, load_configurations, measurements_markdown, findings, pair_status, TUTORIAL_CASES,
)
configs = load_configurations(ROOT)  # Small dictionary; edit actual endpoints/provenance here.

#vLLM set up
configs["rtx_vllm"].update(
hardware="Dell Precision 7740 / Quadro RTX 5000 16 GB",
dtype="float16",
provenance="experiment-tracker/tutorial/provenance/vllm-dell-02.json",
) #vLLM set up end. this hardware does not support BF16, so called out FP16

LIVE_RUNTIMES = {"vllm"} #set()  # e.g. {'llama_cpp'}; one resident engine at a time.
RUN_NIM = False
RUN_EDGE = False

def show(results, trade_off, decision):
    display(Markdown(measurements_markdown(results)))
    display(Markdown(findings(results, trade_off, decision)))
    summary_view.update(Markdown(session.summary()))


## 2. Database and evaluation workload

**Question:** What data and questions are held constant?  
**Configuration:** prebuilt synthetic DuckDB; reviewed schema facts; four cases from the fixed 30.  
**Run:** open read-only; inspect the tables, relationships and non-sensitive trial attributes.  
**Measurements:** table inventory and fixed question IDs; no model run in this section.  
**Finding:** execution match tests rows on this database; it does not certify SQL semantics.


In [3]:
from clinical_trial_mvp.data.schema import RELATIONSHIPS
from clinical_trial_mvp.evaluation.dataset import load_cases
database = ROOT / 'data/db/clinical_trials.duckdb'
assert database.is_file(), 'Run the setup script; do not rebuild data inside the notebook.'
with duckdb.connect(str(database), read_only=True) as db:
    tables = [row[0] for row in db.execute('SHOW TABLES').fetchall()]
    sample = db.execute('SELECT phase, indication, target_enrollment FROM trials LIMIT 3').fetchall()
display(Markdown('**Tables:** ' + ', '.join(tables)))
display(Markdown('**Relationships:**\n\n' + '\n'.join('- ' + r for r in RELATIONSHIPS)))
print('Synthetic trial sample (no subject identifiers):', sample)
cases = [c for c in load_cases(ROOT / 'evals/evaluations.json') if c['id'] in TUTORIAL_CASES]
display(Markdown('\n'.join(f"- **{c['id']}**: {c['question']}" for c in cases)))


**Tables:** adverse_events, data_queries, lab_results, protocol_deviations, screen_failures, site_monitoring, sites, subjects, trials, visits

**Relationships:**

- sites.trial_id -> trials.trial_id
- subjects.site_id -> sites.site_id
- visits.subject_id -> subjects.subject_id
- adverse_events.subject_id -> subjects.subject_id
- screen_failures.subject_id -> subjects.subject_id
- screen_failures.site_id -> sites.site_id
- protocol_deviations.subject_id -> subjects.subject_id (nullable)
- protocol_deviations.site_id -> sites.site_id
- lab_results.subject_id -> subjects.subject_id
- lab_results.visit_id -> visits.visit_id
- data_queries.subject_id -> subjects.subject_id
- data_queries.site_id -> sites.site_id
- data_queries.visit_id -> visits.visit_id (nullable)
- site_monitoring.site_id -> sites.site_id

Synthetic trial sample (no subject identifiers): [('Phase I', 'Rheumatoid Arthritis', 118), ('Phase IV', 'Type 2 Diabetes', 116), ('Phase IV', 'Type 2 Diabetes', 127)]


- **eval_001**: How many subjects are Enrolled or Randomized by treatment arm?
- **eval_002**: How many subjects have enrolled at each site?
- **eval_005**: How many scheduled visits were missed last month?
- **eval_007**: Compare serious adverse-event rates across treatment arms.

Reference SQL supplies execution-match targets. Guardrails reject unsafe SQL and
check question intent before execution. Add human judgements to
`tutorial/semantic_reviews.json`, keyed by case ID and exact SQL hash, to measure
**reviewed semantic correctness**. Unreviewed is unknown, never zero or inferred from matching rows.
The complete [30-question set](../evals/evaluations.json) and [preserved full-result reports](../tutorial/history/README.md)
can be reviewed separately; this compact tutorial keeps one fixed subset.


## 3. MVP1 — Schema-context experiment

**Question:** Does retrieved schema reduce the prompt without losing clinically relevant joins/definitions?  
**Configuration:** one M4 Pro llama.cpp server and one GGUF; change only full versus retrieved schema.  
**Run:** `serve_and_benchmark()` through `session.run()` below.  
**Measurements:** SQL execution, result match, semantic review, prompt tokens and latency.  
**Finding:** embeddings retrieve schema facts; the LLM still receives **text**, hence *retrieved schema*.


In [4]:
mvp1 = [session.run({**configs['mac_llama'], 'profile': profile,
                     'experiment': f'MVP1/{profile}'}, execute='llama_cpp' in LIVE_RUNTIMES)
        for profile in ('full', 'retrieved')]
show(mvp1,
     'Shorter context can omit a necessary join, denominator, or clinical definition.',
     'Prefer retrieved schema only after execution match and semantic review are acceptable.')


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| MVP1/full · plan_only | — | — | — | — | — | — | — | — |
| MVP1/retrieved · plan_only | — | — | — | — | — | — | — | — |

**Observation:** No hardware measurements yet. Planned only; enable this hardware group after starting its server.  
**Trade-off:** Shorter context can omit a necessary join, denominator, or clinical definition.  
**Decision:** Prefer retrieved schema only after execution match and semantic review are acceptable.

## 4. MVP2 — Local serving abstraction

**Question:** What changes when model management/API serving wraps the lower-level runtime?  
**Configuration:** the same GGUF SHA-256, Q4 quantization, context, messages and four questions on M4 Pro.  
**Run:** start llama.cpp, measure it, stop it; prepare that GGUF in Ollama and rerun this cell with only Ollama enabled.  
**Measurements:** the common quality/serving table; inspect token counts and the server-derived chat templates.  
**Finding:** this is low-level runtime control versus packaged model management and API serving.


In [5]:
mvp2 = [session.run({**configs[key], 'experiment': f'MVP2/{configs[key]["runtime"]}'},
                    execute=configs[key]['runtime'] in LIVE_RUNTIMES)
        for key in ('mac_llama', 'mac_ollama')]
print(pair_status(mvp2))
show(mvp2,
     'Ollama simplifies model lifecycle; direct llama.cpp exposes more runtime controls. Warm-cache policies may differ.',
     'Keep Mac results as the default tutorial case; optionally repeat on RTX with hardware recorded.')


Pairing pending: measure both configurations before drawing a comparison.


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| MVP2/llama_cpp · plan_only | — | — | — | — | — | — | — | — |
| MVP2/ollama · plan_only | — | — | — | — | — | — | — | — |

**Observation:** No hardware measurements yet. Planned only; enable this hardware group after starting its server.  
**Trade-off:** Ollama simplifies model lifecycle; direct llama.cpp exposes more runtime controls. Warm-cache policies may differ.  
**Decision:** Keep Mac results as the default tutorial case; optionally repeat on RTX with hardware recorded.

## 5. MVP3 — NVIDIA serving engines

**Question:** How do vLLM and SGLang behave on the same Dell GPU under increasing load?  
**Configuration:** identical frozen Hugging Face snapshot/tokenizer and supported precision; **not** the MVP2 GGUF.  
**Run:** deploy one engine at a time; reuse the same harness at concurrency **1, 2, 4**.  
**Measurements:** TTFT, aggregate tok/s, p50/p95 latency, peak device VRAM, and serial Text-to-SQL quality.  
**Finding:** correlate quality and tail latency with concurrency; do not declare a winner from unmeasured rows.

Measure VRAM on the GPU host with one engine resident. The sampler includes model/cache idle allocation,
can miss peaks between samples, and is not interchangeable with a Mac server's process RSS.


In [6]:
mvp3 = [session.run({**configs[key], 'concurrency': n,
                     'experiment': f'MVP3/{configs[key]["runtime"]}/c{n}'},
                    execute=configs[key]['runtime'] in LIVE_RUNTIMES)
        for key in ('rtx_vllm', 'rtx_sglang') for n in (1, 2, 4)]
if platform.system() == 'Darwin':
    for result in mvp3:
        if result['status'] == 'plan_only':
            result.update(status='skipped', reason='Requires the Dell NVIDIA GPU host; unavailable on this Mac.')
print(pair_status(mvp3))
show(mvp3,
     'Batching improves aggregate throughput but can worsen interactive latency and increase VRAM.',
     'Choose a serving configuration only after inspecting quality, p95 latency and available VRAM.')


Same model/tokenizer fingerprint and prompt workload; inspect hardware and launch settings too.


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| MVP3/vllm/c1 | 3/4 | 2/4 | 0/0 reviewed; 4 total | 462 | 0.19 | 2.63/3.13 | 40.67 | — GiB |
| MVP3/vllm/c2 | 3/4 | 2/4 | 0/0 reviewed; 4 total | 462 | 0.24 | 3.07/3.41 | 67.09 | — GiB |
| MVP3/vllm/c4 | 3/4 | 2/4 | 0/0 reviewed; 4 total | 462 | 0.37 | 3.48/4.37 | 105.37 | — GiB |
| MVP3/sglang/c1 · plan_only | — | — | — | — | — | — | — | — |
| MVP3/sglang/c2 · plan_only | — | — | — | — | — | — | — | — |
| MVP3/sglang/c4 · plan_only | — | — | — | — | — | — | — | — |

**Observation:** 3 measured configurations; review the table before choosing a winner.  
**Trade-off:** Batching improves aggregate throughput but can worsen interactive latency and increase VRAM.  
**Decision:** Choose a serving configuration only after inspecting quality, p95 latency and available VRAM.

## 6. MVP4 — NIM deployment (gated)

**Question:** Does a compatible NIM image/profile simplify deployment and operation?  
**Configuration:** image digest, model, exact GPU and profile discovered on that GPU.  
**Run:** `nim_deploy.py discover`, then deploy **only** a profile in the compatible-and-runnable block.  
**Measurements:** readiness time, common quality/performance, VRAM, image/profile provenance.  
**Finding:** an incompatible, low-memory, or unknown profile is recorded as skipped—not a failed speed result.

If compatible profile exists → deploy and benchmark. Otherwise → record GPU/model/profile reason and skip.
NIM packaging may use a different model or precision; without verified identical weights/tokenizer,
its results are a deployment case rather than a controlled engine-speed comparison.


In [7]:
from clinical_trial_mvp.tutorial.nim import nim_gate
import platform
nim_spec = configs['rtx_nim']
profile_path = ROOT / nim_spec['nim_profile']
nim_profile = json.loads(profile_path.read_text()) if profile_path.exists() else {}
allowed, reason = nim_gate(nim_profile, model=nim_spec['model'], hardware=nim_spec['hardware'],
                           enabled=RUN_NIM, system=platform.system())
nim_result = session.run(nim_spec, execute=allowed)
if not allowed:
    nim_result.update(status='skipped', reason=reason)
show([nim_result],
     'Packaged deployment and profile qualification trade flexibility for operability.',
     'Run only a verified runnable profile; otherwise continue with vLLM optimization.')


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| MVP4/NIM · skipped | — | — | — | — | — | — | — | — |

**Observation:** No hardware measurements yet. NIM disabled; deployment/profile compatibility is unmeasured.  
**Trade-off:** Packaged deployment and profile qualification trade flexibility for operability.  
**Decision:** Run only a verified runnable profile; otherwise continue with vLLM optimization.

## 7. Optimization study — keep vLLM fixed

**Question:** Which single change improves useful performance at acceptable quality?  
**Configuration:** one vLLM baseline plus full/retrieved context, supported quantization, and cache/concurrency tuning.  
**Run:** restart with the appropriate snapshot/settings and distinct provenance file for each variant.  
**Measurements:** the same quality, latency, token and memory fields; change one factor at a time.  
**Finding:** skip unsupported quantization; record changed weights/precision as an intentional intervention.

The quantized case requires an explicitly obtained compatible checkpoint derived from the chosen model.
Do not emulate quantization by relabelling the BF16 model. Cache-enabled experiments require a real
server restart with prefix caching enabled; changing a notebook label alone is rejected by provenance checks.


In [8]:
base = configs['rtx_vllm']
variants = {
    'full': {**base, 'profile': 'full'},
    'retrieved': {**base, 'profile': 'retrieved'},
    'quantized': {**base, 'quantization': 'awq', 'provenance': 'experiment-tracker/tutorial/provenance/vllm-awq.json'},
    'prefix-cache': {**base, 'prefix_cache': True, 'provenance': 'experiment-tracker/tutorial/provenance/vllm-cache.json'},
    'concurrency-4': {**base, 'concurrency': 4},
}
LIVE_VARIANTS = set()  # Enable only the variant matching the currently running server.
optimization = [session.run({**spec, 'experiment': f'Optimization/{name}'},
                            execute='vllm' in LIVE_RUNTIMES and name in LIVE_VARIANTS)
                for name, spec in variants.items()]
if platform.system() == 'Darwin':
    for result in optimization:
        if result['status'] == 'plan_only':
            result.update(status='skipped', reason='Requires a supported NVIDIA GPU and matching running server.')
print(pair_status([optimization[1], nim_result]))
show(optimization + ([nim_result] if nim_result['status'] == 'completed' else []),
     'Reduced context, lower precision and prefix reuse each change different accuracy/memory/latency constraints.',
     'Keep only supported changes with acceptable quality; compare NIM directly only if weight identity is verified.')


Pairing pending: measure both configurations before drawing a comparison.


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| Optimization/full · plan_only | — | — | — | — | — | — | — | — |
| Optimization/retrieved · plan_only | — | — | — | — | — | — | — | — |
| Optimization/quantized · plan_only | — | — | — | — | — | — | — | — |
| Optimization/prefix-cache · plan_only | — | — | — | — | — | — | — | — |
| Optimization/concurrency-4 · plan_only | — | — | — | — | — | — | — | — |

**Observation:** No hardware measurements yet. Planned only; enable this hardware group after starting its server.  
**Trade-off:** Reduced context, lower precision and prefix reuse each change different accuracy/memory/latency constraints.  
**Decision:** Keep only supported changes with acceptable quality; compare NIM directly only if weight identity is verified.

## 8. ExecuTorch portability case — separate from server throughput

**Question:** Where does an edge runtime fit when single-device portability matters?  
**Configuration:** a local supported Llama 3.2 1B checkpoint exported to XNNPACK `.pte`; matching official runner/tokenizer.  
**Run:** export outside this notebook with `scripts/tutorial/export_edge.py`, then run the short case below.  
**Measurements:** startup, model+tokenizer footprint, single-request latency, runner RSS, and reviewed operator fallback.  
**Finding:** edge deployment optimizes a different boundary from concurrent vLLM-style serving.

Runner observer fields provide startup/inference timing when available. Missing fields stay unknown.
Operator fallback requires a matching runtime trace review (ETDump/ETRecord); delegation at export is
not equivalent to runtime fallback. The small model is not ranked against larger clinical models.


In [9]:
from clinical_trial_mvp.tutorial.edge import edge_portability, edge_table
edge = edge_portability({
    'hardware': 'local CPU / XNNPACK', 'runner': 'models/edge/llama_main',
    'model': 'models/edge/llama3_2_1b.pte', 'tokenizer': 'models/edge/tokenizer.model',
}, execute=RUN_EDGE, root=ROOT)
if not RUN_EDGE:
    edge.update(status='skipped', reason='Matching ExecuTorch PTE and runner are not installed/configured.')
display(Markdown(edge_table(edge)))
display(Markdown(findings([edge],
    'Portability and a smaller footprint may trade away model capacity and serving concurrency.',
    'Use edge deployment for an appropriate local workload; do not infer clinical parity from this case.')))
session.results = [r for r in session.results if r['experiment'] != edge['experiment']] + [edge]
summary_view.update(Markdown(session.summary()))


| Status | Startup s | Model+tokenizer MiB | Single request s | Operator fallback count |
|---|---|---|---|---|
| skipped | — | — | — | — |

**Observation:** No hardware measurements yet. Matching ExecuTorch PTE and runner are not installed/configured.  
**Trade-off:** Portability and a smaller footprint may trade away model capacity and serving concurrency.  
**Decision:** Use edge deployment for an appropriate local workload; do not infer clinical parity from this case.

## 9. Optional DLSIM correlation — methodology-only placeholder

**Question:** Can simulation predict the measured RTX 5000 trends?  
**Configuration:** choose 4–6 **measured** configurations with fixed model, token lengths, concurrency and hardware.  
**Run:** no simulator commands, access assumptions, internal artifacts, or internal results are included here.  
**Measurements:** if approved evidence later becomes available, compare measured versus simulated latency/tok/s,
relative error, calibration conditions, and simulator coverage.  
**Finding:** no simulator validation or correlation has been performed.

**Observation:** placeholder; no simulation measurements.  
**Trade-off:** simulation can explore configurations only to the extent its hardware/workload assumptions hold.  
**Decision:** retain as an optional internal extension until access and publication approval exist.

The story: **improve the context → change serving → optimize a validated path → examine portability and simulation.**
